# MERA-XQUAD — Bảng so sánh (§6.3) + phân tích (§7.2)
Gom mọi `results/*.json` (baselines + student + ablation) → Table 1–8.


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Settings -> bật GPU + Internet. Code lấy trực tiếp từ GitHub (như các notebook khác).
!git clone -q https://github.com/hotuyen21pt/MultiLABSA.git
!pip install -q -r MultiLABSA/requirements-kaggle.txt scipy


In [ ]:
import os, fnmatch
def find_dir_by_name(names, root='/kaggle', required_files=None):
    fallback = None
    for dp, _d, files in os.walk(root, followlinks=True):
        if os.path.basename(dp.rstrip('/\\')) in names:
            if not required_files or any(f in files for f in required_files):
                return dp
            fallback = fallback or dp
    return fallback
def find_file(patterns, root='/kaggle'):
    for dp, _d, files in os.walk(root, followlinks=True):
        for fn in files:
            if any(fnmatch.fnmatch(fn, p) for p in patterns):
                return os.path.join(dp, fn)
    return None

GEN_MODEL_DIR   = find_dir_by_name({'hotel-mt5-asqp','hotel_mt5_asqp'}, required_files=['model.safetensors'])
EXTRACTIVE_CKPT = find_file(['extractive_teacher.pt'])
UNLABELED_CSV   = find_file(['hotel_review_merged.csv','hotel_review*_lang.csv'])
# gold labeled: tìm thư mục chứa test.json (dataset hoặc trong repo nếu data_final được commit)
LABELED_DIR = find_dir_by_name({'hamos26'}, required_files=['test.json']) \
              or '/kaggle/working/MultiLABSA/data_final/labeled_data/hamos26'
RESULTS = '/kaggle/working/results'; os.makedirs(RESULTS, exist_ok=True)
print('GEN_MODEL_DIR  :', GEN_MODEL_DIR)
print('EXTRACTIVE_CKPT:', EXTRACTIVE_CKPT)
print('UNLABELED_CSV  :', UNLABELED_CSV)
print('LABELED_DIR    :', LABELED_DIR)
assert GEN_MODEL_DIR, 'Thiếu hotel-mt5-asqp (Add Input dataset).'
assert os.path.exists(os.path.join(LABELED_DIR,'test.json')), 'Thiếu gold test.json (Add Input labeled data hoặc commit data_final).'


In [ ]:
%cd /kaggle/working/MultiLABSA


## (tuỳ chọn) Audit dữ liệu (P0, §5.1) — Table 1


In [ ]:
!python -m experiments.data_prep.audit --labeled_dir "{LABELED_DIR}"


## Dựng toàn bộ bảng so sánh


In [ ]:
!python -m evaluation.make_tables --results_dir {RESULTS} \
  --out /kaggle/working/tables.md --json /kaggle/working/tables.json
print(open('/kaggle/working/tables.md', encoding='utf-8').read())


## Error analysis (§7.2) — ví dụ trên predictions MERA


In [ ]:
import json
from experiments.common.data import load_gold_split
from evaluation.run_eval import align
from evaluation.error_analysis import analyze
reviews, gold, langs = load_gold_split(LABELED_DIR, 'test')
preds = json.load(open('/kaggle/working/preds_mera.json', encoding='utf-8'))
pred_q = align(reviews, gold, preds)
print(json.dumps(analyze(gold, pred_q, langs), ensure_ascii=False, indent=2))


## Kiểm định thống kê (§7.3) — MERA vs M1


In [ ]:
from experiments.common.metrics import exact_quad_f1
from experiments.common.significance import paired_bootstrap
a = align(reviews, gold, json.load(open('/kaggle/working/preds_m1.json', encoding='utf-8')))
b = align(reviews, gold, json.load(open('/kaggle/working/preds_mera.json', encoding='utf-8')))
print(paired_bootstrap(gold, a, b, exact_quad_f1, n_resamples=1000))
